<a href="https://colab.research.google.com/github/er-bubbles/erbeggs-Programming-BigDataAnalytics/blob/main/3_02_key_value_stores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1vv_PsWBnUJwSCkwKDoJAC-vXjtaEA4Ts)

# 3.02 Key-value Stores with TinyDB
This tutorial gives a basic introduction to working with key-value (KV) stores (or document DBs). We will be working with [TinyDB](https://tinydb.readthedocs.io/en/latest/index.html), an in-memory Python database, which is particularly attractive here as it is, as the name suggests, pretty small and lightweight.

We will begin with the relevant installs:

In [1]:
!pip install tinydb
!pip install faker
!pip install python-lorem

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 10.8 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for python-lorem: filename=python_lorem-1.3.0.post3-py3-none-any.whl size=9166 sha256=508f491fd89910e9d685233075d393801a0c9cf73ff6aa7c55b622c5ba6e1ebd
  Stored in directory: /root/.cache/pip/wheels/9d/3a/d2/ba74831189552ab46d4b3d52d3ef86e4f06feef77b0a35d0ea
Successfully built python-lorem


As you may infer from the pacakages installed, we will run something similar to one of our DuckDB examples - specifically building a database using fake data generated by Faker:

In [2]:
import random
from faker import Faker
import pandas as pd
from lorem import paragraph
import itertools

fake = Faker()

def get_person():
  person = {}
  person['id'] = random.randrange(1000,9999999999999)
  person['first_name'] = fake.first_name()
  person['last_name'] = fake.last_name()
  person['email'] = fake.unique.ascii_email()
  person['company'] = fake.company()
  person['phone'] = fake.phone_number()
  person['review'] = list(itertools.islice(paragraph(count=1), 1))
  return person

personlist = []
for x in range(100):
  personlist.append(get_person())

df = pd.DataFrame.from_dict(personlist)
df.head()

,id,first_name,last_name,email,company,phone,review
0,8117987063420,Tammy,Reyes,teresaedwards@jones.net,Medina Ltd,001-729-407-3025x3260,[Culpa ullamco sed consectetur. Sed cillum min...
1,1698389189473,Adam,Chen,donaldpope@mendoza.com,Foster Ltd,(805)656-0936,[Sunt exercitation nulla magna. Sed sunt repre...
2,6082092692281,Todd,Baker,emilyhall@gmail.com,Guzman-Crane,(840)846-1004,[Pariatur dolore laborum dolore. Elit sed volu...
3,2033187583783,Kristin,Thomas,svazquez@gmail.com,Jordan Inc,(885)487-2820,[Irure irure do ad ullamco. Eiusmod consequat ...
4,5153687740841,Randall,Williamson,paul40@velasquez.com,Clark-Jones,001-276-634-5044x604,[Culpa dolore nisi commodo labore. Cillum et a...


Everything here is the same except we have also add a text column (using lorem ipsum). As before we have created this as a Pandas dataframe, but like most KV stores, TinyDB prefers data stored as a dictionary:

In [3]:
fake_data = df.to_dict(orient='records')
fake_data

[{'id': 8117987063420,
  'first_name': 'Tammy',
  'last_name': 'Reyes',
  'email': 'teresaedwards@jones.net',
  'company': 'Medina Ltd',
  'phone': '001-729-407-3025x3260',
  'review': ['Culpa ullamco sed consectetur. Sed cillum minim excepteur ipsum deserunt dolor magna. Consequat consequat non anim et adipiscing esse eu. Voluptate velit voluptate reprehenderit sit consequat ad pariatur. Ullamco voluptate proident irure. Officia elit ullamco elit excepteur sunt pariatur, et nostrud consectetur officia aute cupidatat. Consequat laboris irure voluptate deserunt. Ipsum laborum occaecat anim.']},
 {'id': 1698389189473,
  'first_name': 'Adam',
  'last_name': 'Chen',
  'email': 'donaldpope@mendoza.com',
  'company': 'Foster Ltd',
  'phone': '(805)656-0936',
  'review': ['Sunt exercitation nulla magna. Sed sunt reprehenderit pariatur anim. Duis enim id pariatur. Ex ad dolore proident reprehenderit, aliquip et veniam laborum veniam non veniam, aliqua consectetur dolor dolor quis occaecat. Sin

With this transform in place we can load the data into our database. You may note the database itself is specified as JSON format:

In [4]:
from tinydb import TinyDB, Query

db = TinyDB('db.json')

for record in fake_data:
  db.insert(record)

We can check this has worked with a simple Python loop:

In [5]:
for item in db:
  print(item)

{'id': 8117987063420, 'first_name': 'Tammy', 'last_name': 'Reyes', 'email': 'teresaedwards@jones.net', 'company': 'Medina Ltd', 'phone': '001-729-407-3025x3260', 'review': ['Culpa ullamco sed consectetur. Sed cillum minim excepteur ipsum deserunt dolor magna. Consequat consequat non anim et adipiscing esse eu. Voluptate velit voluptate reprehenderit sit consequat ad pariatur. Ullamco voluptate proident irure. Officia elit ullamco elit excepteur sunt pariatur, et nostrud consectetur officia aute cupidatat. Consequat laboris irure voluptate deserunt. Ipsum laborum occaecat anim.']}
{'id': 1698389189473, 'first_name': 'Adam', 'last_name': 'Chen', 'email': 'donaldpope@mendoza.com', 'company': 'Foster Ltd', 'phone': '(805)656-0936', 'review': ['Sunt exercitation nulla magna. Sed sunt reprehenderit pariatur anim. Duis enim id pariatur. Ex ad dolore proident reprehenderit, aliquip et veniam laborum veniam non veniam, aliqua consectetur dolor dolor quis occaecat. Sint velit pariatur consectetu

With our database setup, we can start to query our records. In TinyDB we do this by creating a query object:

In [6]:
User = Query() # query object

db.search(User.first_name == 'Chad') # adapt based on your data

[]

We can also add new data in dictionary/JSON-like format:

In [7]:
db.insert({'id': 123, 'first_name': 'Amir', 'star_sign': 'Dog', 'review': 'I do not speak Latin.'})

101

And retrieve the data as before:

In [8]:
db.search(User.id == 123)

[{'id': 123,
  'first_name': 'Amir',
  'star_sign': 'Dog',
  'review': 'I do not speak Latin.'}]

One thing to note here is that our new record does not follow the schema we may infer from the original dataset (i.e. the original data all used the same columns/fields). Here many of those fields are missing and we have the new field 'star_sign'.

This demonstrates the extra flexibility we get with a KV store over a relational model. We can also query our database to get all records that have a specific field:

In [9]:
db.search(User.star_sign.exists())

[{'id': 123,
  'first_name': 'Amir',
  'star_sign': 'Dog',
  'review': 'I do not speak Latin.'}]

This gives a basic intro into KV (and document) stores. While there are many competing brands/solutions, the common themes are the dictionary-like structure (key-value pairs) and flexibility to accept any fields (keys).